In [ ]:
from dataclasses import dataclass
from typing import List, Optional
import statistics

# Input Models

@dataclass(frozen=True)
class ThermalMetrics:
    """Thermal inputs."""
    rack_inlet_temps: List[float]
    rack_outlet_temps: List[float]
    supply_air_temp: float
    return_air_temp: float

# Output Model

@dataclass(frozen=True)
class OvercoolingAssessmentRecord:
    """Quantifies waste."""
    overcooling_score: float           # 0.0 to 1.0
    reduction_feasibility_hint: bool   # Hint 
    mean_delta_t: float                # Avg temp rise
    thermal_headroom: float            # Safety gap

# Module Implementation

class EnerviaOvercoolingDetector:
    """Overcooling Detector."""

    def __init__(self):
        # Limits
        self.TARGET_INLET_MAX = 27.0 
        self.MIN_SAFE_DELTA_T = 5.0   # Min Delta T

    def evaluate_overcooling(self, thermal_data: ThermalMetrics) -> OvercoolingAssessmentRecord:
        """Evaluate overcooling."""
        
        # Calc Delta T
        avg_inlet = statistics.mean(thermal_data.rack_inlet_temps)
        avg_outlet = statistics.mean(thermal_data.rack_outlet_temps)
        mean_delta_t = avg_outlet - avg_inlet

        # Calc headroom
        current_max_inlet = max(thermal_data.rack_inlet_temps)
        headroom = self.TARGET_INLET_MAX - current_max_inlet

        # Compute score
        score = self._calculate_score(current_max_inlet, mean_delta_t)

        # Determine feasibility
        is_feasible = (headroom > 2.0) and (mean_delta_t > 2.0)

        # Guard zero division
        if not thermal_data.rack_inlet_temps or not thermal_data.rack_outlet_temps:
            return OvercoolingAssessmentRecord(0.0, False, 0.0, 0.0)
        return OvercoolingAssessmentRecord(
            overcooling_score=round(score, 2),
            reduction_feasibility_hint=is_feasible,
            mean_delta_t=round(mean_delta_t, 2),
            thermal_headroom=round(headroom, 2)
        )

    def _calculate_score(self, max_inlet: float, delta_t: float) -> float:
        """Calculate score."""
        
        # Temp penalty
        temp_factor = max(0, (self.TARGET_INLET_MAX - max_inlet) / 10.0)
        
        # Delta T penalty
        delta_factor = max(0, (self.MIN_SAFE_DELTA_T - delta_t) / self.MIN_SAFE_DELTA_T) if delta_t < self.MIN_SAFE_DELTA_T else 0

        # Weighted score
        raw_score = (temp_factor * 0.7) + (delta_factor * 0.3)
        return min(1.0, raw_score)


# detector = EnerviaOvercoolingDetector()
# metrics = ThermalMetrics(
#     rack_inlet_temps=[21.0, 22.5, 20.8], 
#     rack_outlet_temps=[32.0, 34.0, 31.5],
#     supply_air_temp=18.0,
#     return_air_temp=28.0
# )
# assessment = detector.evaluate_overcooling(metrics)